# 01 — Audit des données et du Gold Standard V1

## Objectif

Reconstituer et auditer les données utilisées dans la V1 afin d'identifier les limites
méthodologiques avant de construire les données ABSA V2.

Cet audit porte notamment sur :

- les sources de données disponibles ;
- la composition du Gold Standard V1 ;
- la relation entre le Gold complet et le benchmark final ;
- les modifications d'annotations d'aspects ;
- la présence de cas multi-aspects ;
- les limites de la représentation V1.

> Ce notebook est un notebook d'audit historique. Il ne modifie pas les artefacts V1.


In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

# Le notebook se trouve dans ABSA-V2/notebooks/
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
GOLD_DIR = DATA_DIR / "gold"
INTERIM_DIR = DATA_DIR / "interim"

FILES = {
    "fr_full": RAW_DIR / "df_fr_full.jsonl",
    "en_full": RAW_DIR / "df_en_full.jsonl",
    "amazon_20k": RAW_DIR / "amazon_reviews_20k.csv",
    "gold_400": GOLD_DIR / "gold_standard_annote.csv",
    "gold_test": GOLD_DIR / "vos_200_avis_manuels_test.csv",
    "pseudo_labels": INTERIM_DIR / "pseudo_labels_fewshot.csv",
    "purified": INTERIM_DIR / "dataset_20k_purifie_v2.csv",
}

for name, path in FILES.items():
    print(f"{name:15} | existe={path.exists()} | {path}")

fr_full         | existe=True | c:\Users\youne\OneDrive\Desktop\ABSA-V2\data\raw\df_fr_full.jsonl
en_full         | existe=True | c:\Users\youne\OneDrive\Desktop\ABSA-V2\data\raw\df_en_full.jsonl
amazon_20k      | existe=True | c:\Users\youne\OneDrive\Desktop\ABSA-V2\data\raw\amazon_reviews_20k.csv
gold_400        | existe=True | c:\Users\youne\OneDrive\Desktop\ABSA-V2\data\gold\gold_standard_annote.csv
gold_test       | existe=True | c:\Users\youne\OneDrive\Desktop\ABSA-V2\data\gold\vos_200_avis_manuels_test.csv
pseudo_labels   | existe=True | c:\Users\youne\OneDrive\Desktop\ABSA-V2\data\interim\pseudo_labels_fewshot.csv
purified        | existe=True | c:\Users\youne\OneDrive\Desktop\ABSA-V2\data\interim\dataset_20k_purifie_v2.csv


In [2]:
datasets = {}

datasets["fr_full"] = pd.read_json(FILES["fr_full"], lines=True)
datasets["en_full"] = pd.read_json(FILES["en_full"], lines=True)
datasets["amazon_20k"] = pd.read_csv(FILES["amazon_20k"])
datasets["gold_400"] = pd.read_csv(
    FILES["gold_400"],
    sep=";",
    encoding="utf-8"
)
datasets["gold_test"] = pd.read_csv(FILES["gold_test"])
datasets["pseudo_labels"] = pd.read_csv(FILES["pseudo_labels"])
datasets["purified"] = pd.read_csv(FILES["purified"])

for name, df in datasets.items():
    print(f"\n--- {name} ---")
    print("Shape :", df.shape)
    print("Colonnes :", list(df.columns))


--- fr_full ---
Shape : (200000, 4)
Colonnes : ['id', 'text', 'label', 'label_text']

--- en_full ---
Shape : (200000, 4)
Colonnes : ['id', 'text', 'label', 'label_text']

--- amazon_20k ---
Shape : (20000, 5)
Colonnes : ['id', 'review_full', 'stars', 'label_text', 'language']

--- gold_400 ---
Shape : (401, 5)
Colonnes : ['id', 'language', 'review_full', 'aspect_manuel', 'sentiment_manuel']

--- gold_test ---
Shape : (198, 5)
Colonnes : ['id', 'language', 'review_full', 'aspect_manuel', 'sentiment_manuel']

--- pseudo_labels ---
Shape : (19600, 4)
Colonnes : ['id', 'review_full', 'language', 'aspects_json']

--- purified ---
Shape : (16089, 5)
Colonnes : ['id', 'review_full', 'language', 'aspects_json', 'binary_labels']


In [3]:
for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(f"DATASET : {name}")
    print(f"{'=' * 60}")

    print("\nShape :")
    print(df.shape)

    print("\nValeurs manquantes :")
    print(df.isna().sum())

    print("\nLignes entièrement vides :")
    print(df.isna().all(axis=1).sum())

    if "id" in df.columns:
        print("\nIDs manquants :")
        print(df["id"].isna().sum())

        print("\nIDs dupliqués :")
        print(df["id"].duplicated().sum())

    text_col = None

    if "review_full" in df.columns:
        text_col = "review_full"
    elif "text" in df.columns:
        text_col = "text"

    if text_col is not None:
        print(f"\nTextes manquants ({text_col}) :")
        print(df[text_col].isna().sum())

        print(f"\nTextes dupliqués ({text_col}) :")
        print(df[text_col].duplicated().sum())


DATASET : fr_full

Shape :
(200000, 4)

Valeurs manquantes :
id            0
text          0
label         0
label_text    0
dtype: int64

Lignes entièrement vides :
0

IDs manquants :
0

IDs dupliqués :
0

Textes manquants (text) :
0

Textes dupliqués (text) :
388

DATASET : en_full

Shape :
(200000, 4)

Valeurs manquantes :
id            0
text          0
label         0
label_text    0
dtype: int64

Lignes entièrement vides :
0

IDs manquants :
0

IDs dupliqués :
0

Textes manquants (text) :
0

Textes dupliqués (text) :
109

DATASET : amazon_20k

Shape :
(20000, 5)

Valeurs manquantes :
id             0
review_full    0
stars          0
label_text     0
language       0
dtype: int64

Lignes entièrement vides :
0

IDs manquants :
0

IDs dupliqués :
0

Textes manquants (review_full) :
0

Textes dupliqués (review_full) :
3

DATASET : gold_400

Shape :
(401, 5)

Valeurs manquantes :
id                  1
language            1
review_full         1
aspect_manuel       1
sentiment_manuel

## 1. Audit du Gold Standard V1

On vérifie ici la taille réelle du Gold, sa relation avec le jeu de test historique,
ainsi que les éventuels doublons ou incohérences structurelles.


In [4]:
gold = datasets["gold_400"].dropna(how="all").copy()
gold_test = datasets["gold_test"].copy()

gold_ids = set(gold["id"])
test_ids = set(gold_test["id"])

print("Gold réel :", len(gold))
print("Gold test :", len(gold_test))

print("\nIDs du Gold test présents dans Gold complet :")
print(len(test_ids & gold_ids))

print("\nIDs du Gold test absents du Gold complet :")
print(len(test_ids - gold_ids))

print("\nGold non présent dans le test :")
print(len(gold_ids - test_ids))

Gold réel : 400
Gold test : 198

IDs du Gold test présents dans Gold complet :
198

IDs du Gold test absents du Gold complet :
0

Gold non présent dans le test :
202


In [5]:
# Gold qui n'appartient pas au fichier de test
gold_hors_test = gold[~gold["id"].isin(test_ids)].copy()

print("Gold hors test :", len(gold_hors_test))

# Recherche de doublons par texte dans ces 202 lignes
duplicates_text = gold_hors_test[
    gold_hors_test.duplicated(subset=["review_full"], keep=False)
].sort_values("review_full")

print("\nNombre de lignes impliquées dans des doublons de texte :")
print(len(duplicates_text))

print("\nNombre de textes distincts dupliqués :")
print(
    duplicates_text["review_full"].nunique()
)

# Afficher un échantillon des doublons pour comprendre ce qui s'est passé
display(
    duplicates_text[
        [
            "id",
            "language",
            "review_full",
            "aspect_manuel",
            "sentiment_manuel"
        ]
    ].head(20)
)


Gold hors test : 202

Nombre de lignes impliquées dans des doublons de texte :
0

Nombre de textes distincts dupliqués :
0


,id,language,review_full,aspect_manuel,sentiment_manuel


In [6]:
gold = datasets["gold_400"].dropna(how="all").copy()
gold_test = datasets["gold_test"].copy()

print("=== GOLD COMPLET ===")
print("Nombre :", len(gold))

print("\nLangues :")
print(gold["language"].value_counts())

print("\nSentiments :")
print(gold["sentiment_manuel"].value_counts())

print("\nCombinaisons d'aspects :")
print(gold["aspect_manuel"].value_counts())


print("\n\n=== GOLD TEST ===")
print("Nombre :", len(gold_test))

print("\nLangues :")
print(gold_test["language"].value_counts())

print("\nSentiments :")
print(gold_test["sentiment_manuel"].value_counts())

print("\nCombinaisons d'aspects :")
print(gold_test["aspect_manuel"].value_counts())

=== GOLD COMPLET ===
Nombre : 400

Langues :
language
fr    200
en    200
Name: count, dtype: int64

Sentiments :
sentiment_manuel
2.0    175
0.0    143
1.0     82
Name: count, dtype: int64

Combinaisons d'aspects :
aspect_manuel
Qualité      274
Prix          49
Service       41
Livraison     36
Name: count, dtype: int64


=== GOLD TEST ===
Nombre : 198

Langues :
language
fr    99
en    99
Name: count, dtype: int64

Sentiments :
sentiment_manuel
2.0    87
0.0    71
1.0    40
Name: count, dtype: int64

Combinaisons d'aspects :
aspect_manuel
Qualité                        119
Qualité, Prix                   26
Qualité, Livraison              18
Qualité, Service                10
Livraison, Service               9
Livraison, Qualité               5
Qualité, Livraison, Service      4
Livraison                        4
Qualité, Prix, Livraison         2
Service                          1
Name: count, dtype: int64


## 2. Comparaison Gold complet / benchmark final

L'objectif est de déterminer si les annotations utilisées lors de l'évaluation finale
sont identiques aux annotations initiales, en distinguant les changements de sentiment
des changements d'aspects.


In [7]:
comparison = gold_test.merge(
    gold[
        [
            "id",
            "aspect_manuel",
            "sentiment_manuel"
        ]
    ],
    on="id",
    how="left",
    suffixes=("_test", "_gold")
)

comparison["aspect_identique"] = (
    comparison["aspect_manuel_test"].astype(str).str.strip()
    ==
    comparison["aspect_manuel_gold"].astype(str).str.strip()
)

comparison["sentiment_identique"] = (
    comparison["sentiment_manuel_test"]
    ==
    comparison["sentiment_manuel_gold"]
)

print("Nombre comparé :", len(comparison))

print("\nAspects identiques :")
print(comparison["aspect_identique"].value_counts())

print("\nSentiments identiques :")
print(comparison["sentiment_identique"].value_counts())

print("\nExemples où l'aspect a changé :")

display(
    comparison.loc[
        ~comparison["aspect_identique"],
        [
            "id",
            "aspect_manuel_gold",
            "aspect_manuel_test",
            "sentiment_manuel_gold",
            "sentiment_manuel_test"
        ]
    ].head(10)
)

Nombre comparé : 198

Aspects identiques :
aspect_identique
True     115
False     83
Name: count, dtype: int64

Sentiments identiques :
sentiment_identique
True    198
Name: count, dtype: int64

Exemples où l'aspect a changé :


,id,aspect_manuel_gold,aspect_manuel_test,sentiment_manuel_gold,sentiment_manuel_test
1,fr_0813707,Service,"Livraison, Qualité",0.0,0.0
2,fr_0009274,Service,"Livraison, Service",0.0,0.0
4,fr_0372843,Livraison,"Livraison, Service",0.0,0.0
5,fr_0487366,Service,"Qualité, Livraison, Service",0.0,0.0
6,fr_0749613,Livraison,"Qualité, Prix, Livraison",0.0,0.0
9,fr_0133219,Service,"Livraison, Service",0.0,0.0
10,fr_0580542,Livraison,"Qualité, Livraison",0.0,0.0
14,fr_0915573,Qualité,"Qualité, Livraison",0.0,0.0
16,fr_0973237,Service,Qualité,0.0,0.0
18,fr_0915037,Livraison,"Livraison, Qualité",0.0,0.0


In [8]:
changed = comparison[~comparison["aspect_identique"]].copy()

print(
    changed[
        [
            "aspect_manuel_gold",
            "aspect_manuel_test"
        ]
    ]
    .value_counts()
    .head(20)
)

aspect_manuel_gold  aspect_manuel_test         
Prix                Qualité, Prix                  21
Livraison           Qualité, Livraison              8
Qualité             Qualité, Livraison              8
Service             Qualité                         7
                    Livraison, Service              6
                    Qualité, Service                6
Qualité             Qualité, Prix                   5
Livraison           Livraison, Qualité              4
Qualité             Qualité, Service                4
Livraison           Livraison, Service              2
                    Qualité, Livraison, Service     2
Service             Qualité, Livraison              2
Livraison           Qualité                         2
Service             Livraison, Qualité              1
                    Qualité, Livraison, Service     1
Livraison           Qualité, Prix, Livraison        1
Qualité             Qualité, Prix, Livraison        1
                    Livraison, Ser

In [9]:
print("Valeurs de sentiment dans les 83 cas modifiés :")
print(changed["sentiment_manuel_test"].value_counts().sort_index())

print("\nRépartition par langue :")
print(changed["language"].value_counts())


Valeurs de sentiment dans les 83 cas modifiés :
sentiment_manuel_test
0.0    38
1.0    10
2.0    35
Name: count, dtype: int64

Répartition par langue :
language
fr    44
en    39
Name: count, dtype: int64


## 3. Analyse multi-aspects du benchmark V1

Cette étape mesure la fréquence des avis contenant plusieurs aspects et la distribution
réelle des occurrences d'aspects dans le benchmark final.

Elle permet surtout de vérifier si la représentation V1 peut correctement décrire un avis
où plusieurs aspects expriment des sentiments différents.


In [10]:
gold_test_audit = gold_test.copy()

gold_test_audit["aspects_list"] = (
    gold_test_audit["aspect_manuel"]
    .str.split(",")
    .apply(lambda aspects: [a.strip() for a in aspects])
)

gold_test_audit["nb_aspects"] = (
    gold_test_audit["aspects_list"].apply(len)
)

print("Nombre d'aspects par avis :")
print(
    gold_test_audit["nb_aspects"]
    .value_counts()
    .sort_index()
)

print("\nAvis avec plusieurs aspects :")
print(
    (gold_test_audit["nb_aspects"] > 1).sum()
)

print("\nPourcentage multi-aspects :")
print(
    round(
        (gold_test_audit["nb_aspects"] > 1).mean() * 100,
        2
    ),
    "%"
)

Nombre d'aspects par avis :
nb_aspects
1    124
2     68
3      6
Name: count, dtype: int64

Avis avec plusieurs aspects :
74

Pourcentage multi-aspects :
37.37 %


In [11]:
ASPECTS = ["Qualité", "Prix", "Livraison", "Service"]

aspect_counts = {
    aspect: gold_test_audit["aspects_list"].apply(
        lambda x: aspect in x
    ).sum()
    for aspect in ASPECTS
}

print("=== OCCURRENCES RÉELLES PAR ASPECT ===")

for aspect, count in aspect_counts.items():
    percentage = count / len(gold_test_audit) * 100
    print(f"{aspect:10} : {count:3} ({percentage:.1f} %)")
    print("\n=== ASPECT × LANGUE ===")

for aspect in ASPECTS:
    print(f"\n{aspect}")

    mask = gold_test_audit["aspects_list"].apply(
        lambda x: aspect in x
    )

    print(
        gold_test_audit.loc[mask, "language"]
        .value_counts()
    )

=== OCCURRENCES RÉELLES PAR ASPECT ===
Qualité    : 184 (92.9 %)

=== ASPECT × LANGUE ===
Prix       :  28 (14.1 %)

=== ASPECT × LANGUE ===
Livraison  :  42 (21.2 %)

=== ASPECT × LANGUE ===
Service    :  24 (12.1 %)

=== ASPECT × LANGUE ===

Qualité
language
en    94
fr    90
Name: count, dtype: int64

Prix
language
en    17
fr    11
Name: count, dtype: int64

Livraison
language
fr    29
en    13
Name: count, dtype: int64

Service
language
fr    12
en    12
Name: count, dtype: int64


In [12]:
print("\n=== ASPECT × SENTIMENT ===")

for aspect in ASPECTS:
    print(f"\n{aspect}")

    mask = gold_test_audit["aspects_list"].apply(
        lambda x: aspect in x
    )

    print(
        gold_test_audit.loc[mask, "sentiment_manuel"]
        .value_counts()
        .sort_index()
    )


=== ASPECT × SENTIMENT ===

Qualité
sentiment_manuel
0.0    62
1.0    38
2.0    84
Name: count, dtype: int64

Prix
sentiment_manuel
0.0     7
1.0     4
2.0    17
Name: count, dtype: int64

Livraison
sentiment_manuel
0.0    24
1.0     3
2.0    15
Name: count, dtype: int64

Service
sentiment_manuel
0.0    15
1.0     5
2.0     4
Name: count, dtype: int64


In [13]:
multi = gold_test_audit[
    gold_test_audit["nb_aspects"] > 1
].copy()

with pd.option_context("display.max_colwidth", 150):
    display(
        multi[
            [
                "id",
                "language",
                "review_full",
                "aspect_manuel",
                "sentiment_manuel"
            ]
        ].head(10)
    )


,id,language,review_full,aspect_manuel,sentiment_manuel
1,fr_0813707,fr,Bof\r\n\r\npiles livrés avec l'emballage d'origine ouvert! Cela met le doute sur les piles … !!!!,"Livraison, Qualité",0.0
2,fr_0009274,fr,Où est ma commande passée il y’a 1mois\r\n\r\nJ’attend toujours ma commande !!!!,"Livraison, Service",0.0
4,fr_0372843,fr,Jamais recu\r\n\r\nJe mets 1 etoile car il faut en mettre une mais je ne peux pas evaluer cet article car malheureusement je ne l'ai jamais recu. ...,"Livraison, Service",0.0
5,fr_0487366,fr,"Neuf?!??\r\n\r\nAchat neuf, produit reçu dans un emballage découpé au cutter et agrafé. Le joint est poussiéreux et vieux. La date de fabrication ...","Qualité, Livraison, Service",0.0
6,fr_0749613,fr,Boîtes défoncées\r\n\r\nBon qualité prix mais emballage les boîtes défoncées,"Qualité, Prix, Livraison",0.0
9,fr_0133219,fr,Arnaque\r\n\r\nJe ne l’ai est jamais reçu je les ai contacter il m’ont dit d’ici 10 jours 2 fois je n’ai jamais reçu,"Livraison, Service",0.0
10,fr_0580542,fr,Barrière arrivée tordue et deux trop grande\r\n\r\nJ’ai acheté cette extension de barrière pour protéger mon petit garçon des chutes. La barrière ...,"Qualité, Livraison",0.0
14,fr_0915573,fr,Je suis déçu...\r\n\r\nLes produits coule dans leurs emballage en étant bien fermé ... Déçu,"Qualité, Livraison",0.0
18,fr_0915037,fr,"bon transfo pas assez emballé\r\n\r\nemballage écrasé et déformation du boitier. Sinon, bonne qualité","Livraison, Qualité",0.0
20,fr_0250930,fr,"Bof\r\n\r\nBeaucoup de dessins, très peu de lecture... Le cadeau a fait son effet (une petite fan ravie), mais une fois lu donc à peine quelques m...","Qualité, Prix",0.0


# Checkpoint — Audit des données et du Gold Standard V1

## Objectif

Cet audit vise à comprendre les limites méthodologiques des données utilisées dans ABSA V1 avant toute nouvelle pseudo-labellisation ou tout réentraînement.

L'objectif n'est pas de corriger directement les datasets V1, mais d'identifier ce qui peut être conservé comme baseline et ce qui doit être reconstruit pour ABSA V2.

---

## 1. Sources brutes

Deux datasets Amazon multilingues sont disponibles :

- Français : 200 000 avis
- Anglais : 200 000 avis
- Total : 400 000 avis

Chaque dataset contient :

- `id`
- `text`
- `label`
- `label_text`

Les 5 niveaux de notation sont équilibrés dans les sources disponibles.

### Intégrité observée

- aucun ID manquant ;
- aucun ID dupliqué ;
- aucun texte manquant ;
- 388 textes dupliqués dans le dataset FR ;
- 109 textes dupliqués dans le dataset EN.

Ces 400 000 avis constituent la source brute à préserver pour ABSA V2.

---

## 2. Sous-échantillon V1

Le dataset V1 `amazon_reviews_20k.csv` contient :

- 20 000 avis ;
- aucun ID dupliqué ;
- aucun texte manquant ;
- 3 textes dupliqués.

La sélection V1 a principalement permis de contrôler la langue et la distribution des étoiles.

Les 400 000 avis n'étant pas annotés par aspect, cette méthode ne permettait pas de contrôler directement la distribution de :

- Qualité ;
- Prix ;
- Livraison ;
- Service.

Il n'est cependant pas encore démontré que la distribution naturelle de ces aspects dans les 400 000 avis constitue elle-même un problème.

Cette question devra être mesurée sur un nouvel échantillon annoté correctement avant d'envisager une stratégie d'équilibrage ou d'enrichissement.

---

## 3. Gold Standard V1

Le fichier Gold contient physiquement 401 lignes, dont une ligne entièrement vide.

Le nombre réel d'annotations est donc :

**400 avis.**

Répartition linguistique :

- FR : 200
- EN : 200

Répartition du sentiment :

- négatif : 143
- neutre : 82
- positif : 175

Le neutre représente donc :

**82 / 400 = 20,5 % du Gold initial.**

Cette proportion montre que la classe Neutre n'était pas inexistante dans les annotations initiales.

Sa faible présence ultérieure dans les pseudo-labels ne permet donc pas, à elle seule, de conclure que cette classe est inutile.

---

## 4. Construction du benchmark final V1

Le fichier de test final contient :

**198 avis**

avec :

- FR : 99
- EN : 99

Tous les 198 IDs appartiennent au Gold initial.

Cependant, les annotations d'aspect ont évolué entre le Gold initial et le benchmark final.

Comparaison sur les 198 IDs :

- sentiment identique : 198 / 198 ;
- aspect identique : 115 / 198 ;
- aspect modifié : 83 / 198, soit environ 41,9 %.

Cette évolution s'explique par une révision ultérieure des annotations visant notamment à ajouter des aspects secondaires manquants.

Une partie des annotations V1 a été réalisée ou révisée avec l'assistance d'un LLM.

Le benchmark V1 doit donc être conservé comme artefact historique et baseline, mais ne doit pas être considéré automatiquement comme une vérité terrain définitive pour V2.

---

## 5. Multi-aspects

Dans les 198 avis du benchmark final :

- 124 contiennent un seul aspect ;
- 68 contiennent deux aspects ;
- 6 contiennent trois aspects.

Soit :

**74 / 198 = 37,37 % d'avis multi-aspects.**

Le multi-aspect est donc fréquent et ne peut pas être traité comme un cas marginal.

---

## 6. Distribution réelle des aspects dans le benchmark final

Après expansion des annotations multi-aspects :

| Aspect | Occurrences | % des 198 avis |
|---|---:|---:|
| Qualité | 184 | 92,9 % |
| Livraison | 42 | 21,2 % |
| Prix | 28 | 14,1 % |
| Service | 24 | 12,1 % |

Le benchmark final V1 est donc fortement dominé par l'aspect Qualité.

En particulier, l'évaluation de Service repose sur un nombre relativement faible d'exemples positifs.

Cela rend les performances par aspect, notamment celles de Service, plus difficiles à interpréter avec précision.

Cette distribution ne permet cependant pas de conclure que les 400 000 avis bruts possèdent la même distribution.

---

## 7. Problème structurel principal identifié

La limitation la plus importante du Gold V1 concerne la représentation du sentiment.

La structure utilisée est essentiellement :

`avis -> liste d'aspects + sentiment unique`

Exemple :

`Qualité, Livraison -> négatif`

Or plusieurs avis observés expriment des sentiments différents selon les aspects.

Exemple réel observé :

> "bon transfo pas assez emballé [...] Sinon, bonne qualité"

L'information ABSA attendue est plutôt :

- Qualité -> positif
- Livraison -> négatif

Un autre exemple contient :

- problème de qualité ;
- retard de livraison ;
- SAV efficace.

Un sentiment global unique ne peut donc pas représenter correctement les trois relations aspect-sentiment.

Cette limitation est indépendante du modèle utilisé.

---

## 8. Schéma cible à tester pour ABSA V2

La représentation V2 doit associer une polarité indépendamment à chaque aspect.

Schéma conceptuel :

| Aspect | Valeur possible |
|---|---|
| Qualité | absent / négatif / neutre / positif |
| Prix | absent / négatif / neutre / positif |
| Livraison | absent / négatif / neutre / positif |
| Service | absent / négatif / neutre / positif |

Cette représentation permet notamment :

- plusieurs aspects par avis ;
- des sentiments contradictoires entre aspects ;
- de distinguer `absent` de `neutre` ;
- de construire séparément un dataset de détection d'aspects et un dataset de classification du sentiment.

La présence de `Neutre` dans ce schéma ne signifie pas encore qu'il sera définitivement conservé dans le modèle final.

Sa pertinence devra être évaluée expérimentalement.

---

## 9. Ce qui est conservé de V1

Les éléments suivants ne doivent pas être supprimés :

- les 400 000 avis bruts ;
- `amazon_reviews_20k.csv` ;
- le Gold V1 ;
- le benchmark V1 ;
- les pseudo-labels Qwen ;
- le dataset après Cleanlab ;
- les notebooks V1 ;
- les modèles XLM-RoBERTa V1 ;
- les seuils V1 ;
- le prompt Qwen V1.

Ils constituent des artefacts historiques nécessaires pour reproduire et comparer V1 avec V2.

En revanche, les datasets annotés/pseudo-annotés de V1 ne seront pas utilisés automatiquement comme vérité terrain V2.

---

## 10. Orientation retenue pour V2

ABSA V2 repart des **400 000 avis bruts** pour reconstruire progressivement une chaîne de données plus fiable.

Aucune décision n'est encore prise concernant :

- l'équilibrage des aspects ;
- la taille finale du nouveau Gold ;
- la conservation définitive de Neutre ;
- le choix du teacher ;
- Qwen vs Llama vs autre LLM ;
- Cleanlab ;
- l'architecture finale des modèles.

Ces décisions devront être basées sur des expériences.

L'annotation V2 pourra être fortement assistée par un LLM performant (notamment ChatGPT), avec un protocole d'annotation explicite et des contrôles ciblés plutôt qu'en supposant qu'une annotation humaine ou LLM constitue automatiquement une vérité parfaite.

---

## 11. Prochaine étape

Construire un **petit échantillon pilote reproductible à partir des 400 000 avis bruts**.

Ce pilote servira à :

1. définir et tester le nouveau protocole d'annotation ABSA ;
2. vérifier la pertinence de `absent / négatif / neutre / positif` ;
3. mesurer approximativement la fréquence naturelle des quatre aspects ;
4. mesurer la fréquence des avis multi-aspects ;
5. observer les différences FR/EN ;
6. identifier les cas ambigus ;
7. décider ensuite, sur données observées, de la taille et de la stratégie de construction du Gold V2.

Aucun entraînement, pseudo-labellisation massive ou équilibrage artificiel ne doit être lancé avant cette validation.

## Conclusion de l'audit

L'audit montre que les artefacts V1 restent utiles comme **baseline historique**, mais que
leur représentation ne constitue pas une vérité terrain suffisante pour ABSA V2.

La décision retenue est donc de conserver les données et résultats V1 pour comparaison,
tout en reconstruisant un Gold V2 où chaque aspect possède indépendamment son propre état :

`absent / négatif / neutre / positif`.

La construction de ce nouveau Gold est réalisée dans le notebook
`02_gold_v2_construction.ipynb`.
